# Step 4: Treatment Exposure Definition

**Study:** Synthetic RWE Diabetes Treatment Study  
**Objective:** Define treatment exposure groups for comparative effectiveness analysis

**Treatment arms:**
- **Group A:** Metformin users
- **Group B:** Other diabetes medication users (insulin, GLP-1, etc.)

**Input:** `study_population.csv` (Step 2) + `medications.csv`  
**Outputs:** Exposure dataset, treatment patterns, summary tables, figures, HTML report

## Exposure definition (observational RWE)

| Element | Definition |
| --- | --- |
| **Index treatment date** | First diabetes medication exposure after T2DM diagnosis |
| **Baseline period** | 365 days before treatment initiation |
| **Follow-up period** | From initiation through 730 days (censored at death/last Rx) |
| **Treatment switch** | Post-index prescription of a different medication class |
| **Multiple medication user** | Both metformin and non-metformin diabetes drugs during follow-up |

Medication classification uses keyword matching on the `DESCRIPTION` field in `medications.csv`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "synthea_csv"
REPORT_DIR = PROJECT_ROOT / "reports"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from treatment_exposure import (
    run_treatment_exposure,
    MEDICATION_CLASS_RULES,
    TREATMENT_GROUP_METFORMIN,
    TREATMENT_GROUP_OTHER,
)

## Medication classification rules

Documented mapping from medication description to drug class and study treatment group.

In [ ]:
rules = pd.DataFrame(
    [{"medication_class": name, "pattern": pat.pattern} for name, pat in MEDICATION_CLASS_RULES]
)
display(rules)

In [ ]:
result = run_treatment_exposure(DATA_DIR, REPORT_DIR)

exposure = result.exposure_dataset
patterns = result.treatment_patterns
summary = result.group_summary

## Exposure dataset and treatment patterns

In [ ]:
print("Exposure windows (first 5 patients):")
display(
    exposure[
        ["patient_id", "treatment_group", "treatment_start_date",
         "baseline_start_date", "followup_end_date", "exposure_category"]
    ].head()
)

print("\nTreatment switching patterns:")
display(
    patterns[
        ["patient_id", "initial_treatment", "subsequent_treatment",
         "switch_indicator", "multiple_medication_indicator"]
    ].head(10)
)

## Treatment group summary

In [ ]:
display(summary)

print(f"\nSwitch rate: {patterns['switch_indicator'].mean()*100:.1f}%")
print(f"Multiple medication rate: {patterns['multiple_medication_indicator'].mean()*100:.1f}%")

## Visualizations

In [ ]:
from IPython.display import Image, display

for fig_path in result.figure_paths:
    print(fig_path.name)
    display(Image(filename=str(fig_path)))

## Next steps

Treatment exposure definitions feed into:
1. **Propensity score modeling** — adjust for baseline imbalances between groups
2. **Outcome analysis** — ITT (assigned at index) vs as-treated (switching) sensitivity analyses
3. **Exposure windows** — baseline covariates and follow-up outcomes use dates from `exposure_dataset.csv`

Full documentation: `reports/treatment_exposure_report.html`